# Universelles Decoder-Modell 

## Daten laden

Unabhängig vom verwendeten Modell laden wir zunächst die bereits installierten Pakete sowie unsere Textdaten und speichern diese in einem Listenobjekt:

In [1]:
#| echo: false
from transformers import logging
logging.set_verbosity_error()  # Mitteilungen muten, damit Dokuemnt ordentlich bleibt

In [2]:
import pandas as pd
from transformers import pipeline
import json

daten = pd.read_csv("data/example_dataset.csv")
datenliste = list(daten["text"])

daten

,text
0,Die orthodoxe Gemeinde feiert heute #Ostern. I...
1,Vielen Dank Gregor Rutz für die Unterstützung ...
2,Wird ja immer schlimmer mit den Intoleranten d...
3,"Liebe Junge, geht wählen. Kann ja nicht sein, ..."
4,‘— Das war unser EU-Wahl-Abschluss der SP– mit...
5,Aufschlussreicher Blick hinter die Mauern! Tol...
6,"Denn ein Hard-Brexit würde der #EU, aber noch ..."
7,Familienbonus: Es ist erstaunlich wie viele Me...
8,Integration passiert vor Ort. Deshalb arbeitet...
9,Ich frage mich: Wer ist als nächstes dran Frau...


## Klassifikation

Anders als bei den vorigen Ansätzen müssen wir bei einem universellen Decoder-Modell einen Prompt entwickeln, auf dessen Basis das Modell Text generiert (d. h. die Antwort auf die Codieranfrage), idealerweise in einem maschinenlesbaren Format. Der Prompt könnte bspw. folgendermaßen lauten: *“Du bist ein trainierter Assistent für Inhaltsanalyse, der die allgemeine Stimmung beziehungsweise Tonalität von Texten analysiert. Antworte immer präzise im JSON-Format mit sentiment (positiv, neutral, negativ) und reasoning (eine Begründung auf Deutsch). Gib ausschließlich den JSON-Response, beginnend mit '{' und endend mit '}', mit diesen zwei Parametern zurück.”* Diesen Prompt wenden wir nun iterativ auf jeden Text in unserem Datensatz an. Dies geht am einfachsten über eine *for*-Schleife.

Über den Parameter `max_new_tokens` können wir die Länge des generierten Outputs beschränken, `do_sample` sollten wir i.d.R. auf “False” setzen, um deterministische und somit replizierbare Antworten zu erhalten. Zu beachten ist außerdem, dass große Sprachmodelle wie bspw. das hier verwendete LLama-Modell 3.1 (`model="meta-llama/Meta-Llama-3.1-8B-Instruct"`) über Hunggingface zwar prinzipiell frei verfügbar sind, oftmals jedoch eine Authentifizierung erfordern. 

In [3]:
generate = pipeline("text-generation", model="meta-llama/Meta-Llama-3.1-8B-Instruct")
# Leere Liste für den Output
ergebnis = []
# Eine Schleife zur Output-Generierung über jeden Text in der Liste laufen lassen
for text in datenliste:
    instructions = [
        {"role": "system", 
         "content": "Du bist ein trainierter Assistent für Inhaltsanalyse, der die allgemeine Stimmung beziehungsweise Tonalität von Texten analysiert. Antworte immer präzise im JSON-Format mit sentiment (positiv, neutral, negativ) und reasoning (eine Begründung auf Deutsch). Gib ausschließlich den JSON-Response, beginnend mit '{' und endend mit '}', mit diesen zwei Parametern zurück"},
        {"role": "user", "content": text}]
    outputs = generate(instructions, max_new_tokens=256, do_sample=False)
    ergebnis.append({"text": text, "response": outputs[0]["generated_text"][-1]['content']})

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Anschließend können wir die Ergebnisse aus dem JSON-Response extrahieren. Wenn wir möchten, können wir die zusätzliche ausgegebene Begründung ebenfalls abspeichern, um uns diese anzusehen.

In [4]:
def parse_response(response):
    parsed = json.loads(response)
    return parsed['sentiment'], parsed['reasoning']

parsed_data = [(entry['text'], *parse_response(entry['response'])) for entry in ergebnis]
parsed_data = pd.DataFrame(parsed_data, columns=['text', 'sentiment', 'reasoning'])

parsed_data

,text,sentiment,reasoning
0,Die orthodoxe Gemeinde feiert heute #Ostern. I...,positiv,Der Text wünscht ein 'frohes und gesegnetes Fe...
1,Vielen Dank Gregor Rutz für die Unterstützung ...,positiv,"Der Text ist positiv, da er jemanden (Gregor R..."
2,Wird ja immer schlimmer mit den Intoleranten d...,negativ,"Der Text beschreibt eine negative Entwicklung,..."
3,"Liebe Junge, geht wählen. Kann ja nicht sein, ...",negativ,Der Text enthält eine Aufforderung zum Handeln...
4,‘— Das war unser EU-Wahl-Abschluss der SP– mit...,positiv,"Der Text enthält ein Ausrufezeichen, das auf e..."
5,Aufschlussreicher Blick hinter die Mauern! Tol...,positiv,"Der Text enthält positive Wörter wie 'Tolle', ..."
6,"Denn ein Hard-Brexit würde der #EU, aber noch ...",negativ,"Der Text besagt, dass ein Hard-Brexit sowohl d..."
7,Familienbonus: Es ist erstaunlich wie viele Me...,negativ,Die Verwendung des Wortes 'erstaunlich' in die...
8,Integration passiert vor Ort. Deshalb arbeitet...,neutral,Der Text beschreibt eine praktische Zusammenar...
9,Ich frage mich: Wer ist als nächstes dran Frau...,neutral,Die Frage ist neutral formuliert und bezieht s...
